In [2]:
# Librerias y dependencias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn import model_selection
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import ElasticNet
from math import sqrt
from sklearn.metrics import r2_score
from sklearn.linear_model import LassoCV
from numpy import mean
from numpy import std
from numpy import arange
from numpy import absolute
from pandas import read_csv
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import f1_score, accuracy_score
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_error
import sys
import random


In [34]:
'''
    Esta clase permite realizar la codificación en caliente especificameente variables categoricas
'''

class OneHotCoding():
    def __init__(self, df, bin_features):
        self.bin_features = bin_features
        self.df = df
    
    # Metodo para realizar la codificacion dummy a las variables categoricas

    def dummyCodification(self):
        cat_features = self.df.select_dtypes(include = ["object", "category"]).columns
        bin_dataset = self.df[self.bin_features].replace({'SI': 1, 'NO': 0})
        categorical_features = [x for x in cat_features if x not in self.bin_features]
        df_cat = pd.get_dummies(self.df[categorical_features],dtype=int)
        self.df.drop(cat_features, axis = 1, inplace = True)
        df_final = pd.concat([self.df,df_cat,bin_dataset ], axis = 1)
        df_final.to_excel("../Archivos Generados/PipelineResults/dasetOneHot.xlsx")
        print("Ejeción Terminada")
        return df_final


#categorical_transformer = Pipeline(
#    steps=[("OneHotCoding",  OneHotCoding(df,bin_features).dummyCodification())]
#)


class LinearRegession():
    def __init__(self, df, alpha, l1_ratio):
        self.df = df
        self.alpha = alpha
        self.l1_ratio = l1_ratio
    

    def CalcularModeloLR(self):
        # alpha=0.1, l1_ratio=0.97
        Y = self.df.RDT_AJUSTADO.values
        X = self.df.drop(["RDT_AJUSTADO","ID_LOTE"], axis=1).values
        #print("Longitud X: ", X.shape) 
        modelElasticNet = ElasticNet(alpha=self.alpha, l1_ratio=self.l1_ratio, random_state=123)
        model = modelElasticNet.fit(X,Y)
        r_2 = model.score(X,Y)
        # Pedicciones
        yhat = model.predict(X)

        return [model, r_2, yhat]


class CLR():

    def __init__(self, df, yhat):
        self.df = df
        self.yhat = yhat

    
    def calcularMAE(self):
        contador=0
        EPA = 0
        Acumulador = 0
        accepted_average_error= []
        self.df["yhat"]= pd.Series(self.yhat)
        self.df["EA"] = abs(self.df.RDT_AJUSTADO - self.df.yhat)
        self.df_ordely = self.df.sort_values(by=['EA'],ascending=True).reset_index()

        # Calculamos el MAE
        for i in range(len(self.df_ordely)):
            Acumulador = Acumulador + self.df_ordely.loc[i].EA
            EPA = Acumulador/(i+1)
            accepted_average_error.append(EPA)
        
        # Agregamos el promedio al dataset Ordenado.
        self.df_ordely["MAE"] = pd.Series(accepted_average_error)
        #self.df_ordely.to_excel(f"FASE1/DatasetOrdenadoIteraciónesss{contador +1}.xlsx")
        #self.df.to_excel("../Archivos Generados/PipelineResults/DatasetOriginal.xlsx")
        contador=contador+1
        return self.df_ordely
    


def DeleteRecordsGroup(Group, df):
    indexEliminar = list(Group["index"])
    df_new = df[df.index.isin(indexEliminar)== False]
    return df_new


# Retorna el grupo a modificar
def compareCorrelation(CorelacionGruposCalidad, nuevaCorrelation, listaGrupos):
    lista_grupos = listaGrupos
    #print("lista grupos: ",lista_grupos)
    arr =  np.array(CorelacionGruposCalidad) - np.array(nuevaCorrelation)
    position = np.where(arr == np.amin(arr))
    #print("Posision grupo: ",position)
    indexGrupoModificar = position[0][0]
    #print("index: ", indexGrupoModificar)
    lista_grupos.pop(indexGrupoModificar)
    return lista_grupos, indexGrupoModificar


def dataframeNormalized(dataset):
    Y = dataset.RDT_AJUSTADO
    X = dataset.drop(["RDT_AJUSTADO"], axis=1)
    # Nombre columnas de X[Variables independientes]
    X_name_columns= X.columns
    scaler = MinMaxScaler()
    X_normalized = scaler.fit_transform(X.values)
    df_X_normalized = pd.DataFrame(X_normalized, columns= X_name_columns)
    df_normalized = pd.concat([df_X_normalized, Y], axis=1)
    print(df_normalized.shape)
    return df_normalized



# Funcion para Normalizar la Vista minable a exepción de la etiqueta(Variable Objetivo)
def EncoderViewMinable(df):
    new_dataset = df
    norm = MinMaxScaler()
    norm = norm.fit(new_dataset.values[:,:])
    valMin = norm.data_min_
    valMax = norm.data_max_
    dataRange = norm.data_range_
    #df_norm = norm.fit_transform(df.values[:,:-1])

    return [valMin, valMax, dataRange]


def fase1(dataset,Minimum_records,minimum_correlation, MAE_Allowed,additional_average_error):
    group_acepted = []
    correlation_model =[]
    model_acepted = []
    contador = 0
    while (len(dataset) !=0):
        contador = contador+ 1
        print("Tamaño del dataset: ", dataset.shape)
        modellr, r_2, yhat = LinearRegession(dataset, 0.1, 0.97).CalcularModeloLR()
        print("Ajuste del Modelo dataset Completo: ", r_2)
        DatasetOrdely = CLR(dataset,yhat).calcularMAE()
        #DatasetOrdely.to_excel(f"FASE1/DatasetOrdenado{contador}.xlsx")

        try:
            group = DatasetOrdely.loc[DatasetOrdely.MAE < MAE_Allowed]
        except:
             print(f"No se cumple con el criterio de selección, verificar variable MAE_ALLOWED: {MAE_Allowed}")
             # Retornar variables vacias
             break

        print(f"Longitud de grupo {contador}: ", len(group))
        if (len(group) >= Minimum_records):
                print("El grupo cumple minimo de registros")
                group = group.drop(["yhat","EA","MAE"], axis=1)
                dataset = dataset .drop(["yhat","EA"], axis=1)
                group_model, r2_group_mode, yhat_group = LinearRegession(group,0.1,0.97).CalcularModeloLR()
                print(f"R2 del grupo {contador}: ",r2_group_mode)
                if(r2_group_mode >= minimum_correlation):
                    # Elimino los registros para la siguiente iteración
                    dataset = DeleteRecordsGroup(group, dataset)
                    group = group.drop(['index'], axis=1)
                    group_acepted.append(group)
                    model_acepted.append(group_model)
                    correlation_model.append(r2_group_mode)
                    group.to_excel(f"FASE1/GrupoN_{contador}.xlsx")
                    MAE_Allowed = MAE_Allowed + additional_average_error
                else:
                    print("No cumple con la condición de  la correlacion")
                    Orphans = dataset
                    # Se eliminan las variables de trtamiento [yhat, EA]
                    Orphans = Orphans.drop(["yhat", "EA"],axis=1).reset_index(drop=True)
                    print("Logitud Huerfanos: ", Orphans.shape)
                    break  
        else:

            Orphans = dataset
            # Se eliminan las variables de trtamiento [yhat, EA]
            Orphans = Orphans.drop(["yhat", "EA"],axis=1).reset_index(drop=True)
            print("Logitud Huerfanos: ", Orphans.shape)
            # Se guardan los huerfanos en un archivo.
            Orphans.to_excel("FASE1/HuerfanosN.xlsx")
            break
    
    return [group_acepted, model_acepted, correlation_model, Orphans]



def fase2(group_acepted, correlation_model, Orphans):
    # Variables de entrada
    quality_groups = group_acepted.copy()
    correlation_quality_groups = correlation_model.copy()
    new_correlation = []

    group_list = list(range(len(quality_groups)))
    for register in range(len(Orphans)):
        for group in range(len(quality_groups)):
            quality_groups[group] = pd.concat([Orphans.loc[[0]], quality_groups[group]],ignore_index=True)        
            new_model, new_r2, new_yhat = LinearRegession(quality_groups[group],0.1, 0.97).CalcularModeloLR()
            new_correlation.append(new_r2)

        # Compración de las correlaciones
        list_groups_remove, index = compareCorrelation(correlation_quality_groups, new_correlation, group_list)
        #print(f"Grupos Eliminar registro: {list_groups_remove} y indeice del grupo a amntener registro {index}")
        correlation_quality_groups[index] = new_correlation[index]
        new_correlation = []
        group_list = list(range(len(quality_groups)))

        # Eliminacion de Regristro en los demas grupos
        for i in list_groups_remove:
            quality_groups[i].drop([0],axis=0, inplace=True)

        Orphans.drop([0],axis=0, inplace=True)
        Orphans = Orphans.reset_index(drop=True)

    return [quality_groups,correlation_quality_groups, Orphans]




#  Funciones Improvisación
#===================================================================================================

# Construir grupos de Calidad
def BuildGroupsQuality(definitive_groups):
    for i in range(len(definitive_groups)):
        definitive_groups[i]["Grupo"]= (i)

    df = pd.concat(definitive_groups, ignore_index=True)
    df = df.drop(["ID_LOTE"], axis=1)
    return df



'''
Funcion para generar el vector de pesos aleatorio.
Entradas:
w: Vector de pesos w [0,1], igual al numero de caracteristicas normalizadas.
nc: Nunero de ceros que debe contener el vector de pesos [10,20,30,40,50]
'''
def GenerateWeightVector(size):
    w = np.random.uniform(low=0, high=1, size=(size))
    sum = w.sum()
    w = w / sum

    return w



'''
-Función de calidad, que retorna la metrica de calidad asociada a ese vector w especifico
-Se debe tener en cunata la seleccion de la metrica de calidad asociada, para evaluar
el desempeño del algoritmo (Problema de Clasificacion)
Entradas:
    df_norm: Dataset Normalizado.
    wi: Vector de Pesos.
'''

'''
# Dependiendo del vector de pesos me extrae el acuracy - F1 score
def qualityFunction(df_norm, wi,df):
    #print("longitud df_norm: ",len(df_norm))
    #print("Longitud wi: ", len(wi))
    y_pred = []
    for i in range(len(df_norm)):
        vrf = df_norm[i] 
        minDep = sys.float_info.max
        posMinDep = 0
        for j in range(len(df_norm)):
            if i != j:
                ri = wi* np.power((df_norm[j] - vrf), 2)
                dE = np.sum(ri)
                if dE < minDep:
                    posMinDep=j
                    minDep = dE
        #print(posMinDep)
        y_pred.append(df.values[posMinDep][-1])
    qs = accuracy_score(list(df.values[:,-1]), y_pred)
    return qs
'''



# Dependiendo del vector de pesos me extrae el acuracy - F1 score

# df: Datasaset distancia que incluye los atributos con los grupos
def qualityFunction(df_norm, wi,df, lista_modelos, limites_15):
    #print("longitud df_norm: ",len(df_norm))
    #print("Longitud wi: ", len(wi))
    y_pred = []
    for i in range(len(df_norm)):
        vrf = df_norm[i] 
        minDep = sys.float_info.max
        posMinDep = 0
        for j in range(len(df_norm)):
            if i != j:
                ri = wi* np.power((df_norm[j] - vrf), 2)
                dE = np.sum(ri)
                if dE < minDep:
                    posMinDep=j
                    minDep = dE
        print(posMinDep)
        modelo = df.values[posMinDep][-1]
        print("Modelo: ", modelo)
        # caculamos la predicción de c/d registro con el modelo seleccionado
        y_pred = lista_modelos[modelo].predict(df.values[:,:-1][i].reshape(1,-1))[0]
        
        if (y_pred >= limites_15.values[i][2]) & (y_pred <= limites_15.values[i][2]):
            contador = contador + 1
        
    qs = contador/len(df_norm)
    print("Calidad: ", qs)
    return qs



'''
Función para generar la memoria Armonica
Entradas:
    MAC: Tamaño de la  Memoria Armonica
    wi: Vector de Pesos.
    nc: Numero de ceros (Selección de atributos)
'''

def GenerateArmonyMemory(df_norm, MAC,df,lista_modelos, limites_15):
    Lw = []
    for i in range (MAC):
        wi = GenerateWeightVector(174)
        Qs = qualityFunction(df_norm, wi,df,lista_modelos, limites_15)
        wiq = np.append(wi, Qs)
        Lw.append(wiq)

    
    Lw.sort(key=lambda x: x[-1], reverse=True)
    return Lw



def ImprovisationGBHS(PAR, hmn, nc,dataset_normalizado , lmp, HMRC,df,lista_modelos, limites_15):

    df_norm = dataset_normalizado

    # Se genera la memoria Armonica - diferentes tamaños
    np.random.seed(123)
    MA = GenerateArmonyMemory(df_norm,hmn,df,lista_modelos, limites_15)
    P=len(MA[0]) 
    
    
    curvaFitnes = []
    vectorIteration= []
    for i in range (lmp):
        pesosAleatorios = np.random.rand(P-1)
        for j in range(P-1):
            Aleatorio1 = random.random() 
            if (Aleatorio1 < HMRC):
                pma = random.randint(0, hmn-1)
                pesosAleatorios[j]= MA[pma][j]

                Aleatorio2 = random.random()
                if Aleatorio2 < PAR:
                    pesosAleatorios[j] = MA[0][j]
            
            else:
                Aleatorio3 = random.random()
                if Aleatorio3 < nc/P:
                    Aleatorio4 = 0
                else:
                    Aleatorio4 = Aleatorio3/(P-nc)
                
                pesosAleatorios[j] = Aleatorio4
        

        # Normalización de los pesos
        wf = GenerateWeightVector(pesosAleatorios, 0)
        fitnes = qualityFunction(df_norm, wf,df)



        # Remplazo
        if MA[hmn -1][P-1] < fitnes:
            new_register = np.append(wf, fitnes)
            MA[hmn-1] = new_register
            #print("------------------------------------")
            MA.sort(key=lambda x: x[-1], reverse=True)
        

        
        curvaFitnes.append(MA[0][P-1])
        vectorIteration.append(MA[0])
        

    
    
    print("Valor de la curva en la ultima poisción: ",curvaFitnes[-1])
    '''
    dicc = {"vector":vectorIteration,
                "Fitnes": curvaFitnes}
    

    df_new = pd.DataFrame(data=dicc)
    df_new.to_csv(f"ResultadosImprovisacion/GBHS.csv")
    dicc = dict()
    '''
    return [curvaFitnes[-1], vectorIteration[-1]]


# Funcion para Normalizar la Vista minable a exepción de la etiqueta(Variable Objetivo)
# df: es la matriz df.values [] , no incluye la etiqueta del grupo
def NormalizeViewMinable(df,valMin, dataRange):
    dataset_normalizado = np.empty((df.shape[0], df.shape[1]))
    for i in range(df.shape[0]):
        for j in range(df.shape[1]):
            dataset_normalizado[i][j]= (df[i][j] - valMin[j])/dataRange[j]

    return dataset_normalizado




def predictionTest(dataset_test_norm, df_norm, vector_pesos_optmizacion, final_datset_join, list_final_models,dataset_validation):
    df_groups_finally = df_norm.copy()
    minDep = sys.float_info.max
    lista_asignacion_grupos = []
    print("Longitud dataset norm: ",dataset_test_norm.shape[0])
    print("Vista minable : ",df_groups_finally.shape[0])
    posMinDep = 0
    y_pred_test = []
    for z in range(int(dataset_test_norm.shape[0])):
        for k in range(int(df_groups_finally.shape[0])):
            ri = vector_pesos_optmizacion * np.power((dataset_test_norm[z]- df_groups_finally[k]),2) 
            dE = np.sum(ri)
            if dE < minDep:
                posMinDep = k
                minDep=dE
        
        #print("zz, " , z)
        # Grupo seleccionado
        grupoSelected = int(final_datset_join.values[posMinDep][-1])
        lista_asignacion_grupos.append(grupoSelected)   
        #print(grupoSelected)
        #model_final, r2_final, yhat_final = LinearRegession(definitive_groups[i],0.1, 0.97).CalcularModeloLR() 
        psi_predicho = list_final_models[grupoSelected].predict(dataset_validation.values[z].reshape(1,-1))
        #print(psi_predicho)
        y_pred_test.append(psi_predicho[0])

    return [y_pred_test,lista_asignacion_grupos]



def metricasPerformanceCLR(y_true, yhat_test):

    R2 = r2_score(y_true, yhat_test)
    MAE =  mean_absolute_error(y_true, yhat_test)
    print(R2)
    print(MAE)




In [5]:

# Variables Globales
# ==============================================================================
Minimum_records = 64
minimum_correlation= 0.88
MAE_Allowed = 143                                                         #MAE
additional_average_error = 98
contador = 0
definitive_groups = []

#1. Lectura del Dataset Principal
# ==============================================================================

#df = pd.read_excel("../Data/Gold/DatasetFinal.xlsx")
df = pd.read_csv("../Data/Gold/DatasetFinalFP.csv")
#print(df.head())
# Ornial variables list
bin_features =['SEM_TRATADAS','DRENAJE','ALMACENAMIENTO_FINCA','CAP_ENDURE_RASTA','MOTEADOS_RASTA','MOTEADOS_MAS70cm._RASTA',
               'OBSERVA_EROSION_RASTA','OBSERVA_MOHO_RASTA','OBSERVA_RAICES_VIVAS_RASTA','OBSERVA_HOJARASCA_MO_RASTA',
                'SUELO_NEGRO_BLANDO_RASTA','CUCHILLO_PRIMER_HTE_RASTA','CERCA_RIOS_QUEBRADAS_RASTA',
               ]


#2. Codificación de variables categoricas - Dummy
# ============================================================
dataset = OneHotCoding(df,bin_features).dummyCodification()
print("Dimension dataset Original: ", dataset.shape)

dataset_original = dataset.copy()
dataset_vindep = dataset.copy()
dataset_vindep= dataset_vindep.drop(["RDT_AJUSTADO","ID_LOTE"],axis=1)



#2. Encoders [Min, Max, data Range] para aplicar data Normalization
# =================================================================
'''
    Entradas: dataset: vista minable de caracteristicas indepenedientes, exepto la V objetivo
'''
valMin, valMax, dataRange = EncoderViewMinable(dataset_vindep)
print("Longitudes : ", len(valMin), len(valMax), len(dataRange))
# Guardamos los encoders apra posteriores usos
np.savetxt('FASE2/Encoder_ValMin.txt', valMin)
np.savetxt('FASE2/Encoder_dataRange.txt', dataRange)


#3. División de datset Training and Test
# ============================================================
dataset_train, dataset_test = train_test_split(dataset, test_size = 0.1, random_state=92)
print("Longitud Dataset Entrenamiento:",  dataset_train.shape)
dataset_training = dataset_train.copy()


# 4. Construcción de grupos de CalidaD FASE 1
# ============================================================
group_acepted, model_acepted, correlation_model, Orphans = fase1(dataset_training, Minimum_records,minimum_correlation, MAE_Allowed,additional_average_error)
print("---------------- FASE 1---------------------")
print("Correlaciones Iniciales: ", correlation_model)
#print(f"Grupo 1 {len(group_acepted[0])}, Grupo 2: {len(group_acepted[1])}, Grupo 3: {len(group_acepted[2])}")
print("Huerfanos: ", Orphans.shape)


# 5. Construccción de grupos definitivos 
# ============================================================

if len(group_acepted) > 1:
    if len(Orphans) == 0:
        definitive_groups = group_acepted
    
    else:
        # Fase 2, incluir los huerfanos
        print("FASE 2")
        quality_groups, correlation_quality_groups, orphans = fase2(group_acepted,correlation_model, Orphans)
        print("------------------ FASE 2 -------------------")
        print("Correlaciones Finales: ", correlation_quality_groups)
        #print(f"Grupo 1 {len(quality_groups[0])}, Grupo 2: {len(quality_groups[1])}, Grupo 3: {len(quality_groups[2])}")
        print("Huerfanos: ", orphans.shape)
        # Guardamos los grupos Finales
        for c, g in enumerate (quality_groups):
            print(f"Longitud Grupo {c} :  {len(quality_groups[c])} ")
            g.to_excel(f"FASE2/grupo_N{c}.xlsx")
   
        
        for c, value in enumerate (correlation_quality_groups):
            if value < 0.88:
                dataset_group = quality_groups[c]
                # Se aplica el mismo proceso que la fase 1
                group_acepted2, model_acepted2, correlation_model2, Orphans2 = fase1(dataset_group ,Minimum_records,minimum_correlation, MAE_Allowed,additional_average_error)
            else:
                definitive_groups.append(quality_groups[c])


        # Aqui va el proceso de Afinamieno e improvisación 
        
else:
    definitive_groups = Orphans


print("Grupos Definitivos: ", len(definitive_groups))

# Modelos Finales
list_final_models = []
for i in range(len(definitive_groups)):
    model_final, r2_final, yhat_final = LinearRegession(definitive_groups[i],0.1, 0.97).CalcularModeloLR()
    print("R2: ", r2_final)
    list_final_models.append(model_final)
    


grupos_finales = definitive_groups.copy()


C:\Users\germanm\AppData\Local\Temp\ipykernel_14844\1125052729.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  bin_dataset = self.df[self.bin_features].replace({'SI': 1, 'NO': 0})
C:\Users\germanm\AppData\Local\Temp\ipykernel_14844\1125052729.py:19: UserWarning: Pandas requires version '3.0.5' or newer of 'xlsxwriter' (version '3.0.3' currently installed).
  df_final.to_excel("../Archivos Generados/PipelineResults/dasetOneHot.xlsx")


Ejeción Terminada
Dimension dataset Original:  (799, 176)
Longitudes :  174 174 174
Longitud Dataset Entrenamiento: (719, 176)
Tamaño del dataset:  (719, 176)
Ajuste del Modelo dataset Completo:  0.7353483834467243


c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.202e+07, tolerance: 1.352e+05
  model = cd_fast.enet_coordinate_descent(


Longitud de grupo 1:  82
El grupo cumple minimo de registros
R2 del grupo 1:  0.9836604206704304


c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.014e+06, tolerance: 8.108e+03
  model = cd_fast.enet_coordinate_descent(
C:\Users\germanm\AppData\Local\Temp\ipykernel_14844\1125052729.py:161: UserWarning: Pandas requires version '3.0.5' or newer of 'xlsxwriter' (version '3.0.3' currently installed).
  group.to_excel(f"FASE1/GrupoN_{contador}.xlsx")


...

c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.176e+06, tolerance: 9.375e+04
  model = cd_fast.enet_coordinate_descent(
C:\Users\germanm\AppData\Local\Temp\ipykernel_14844\1125052729.py:177: UserWarning: Pandas requires version '3.0.5' or newer of 'xlsxwriter' (version '3.0.3' currently installed).
  Orphans.to_excel("FASE1/HuerfanosN.xlsx")


Longitud de grupo 5:  37
Logitud Huerfanos:  (372, 176)
---------------- FASE 1---------------------
Correlaciones Iniciales:  [0.9836604206704304, 0.9693664058181423, 0.9693428877426256, 0.9986926562798951]
Huerfanos:  (372, 176)
FASE 2


c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.321e+06, tolerance: 9.059e+03
  model = cd_fast.enet_coordinate_descent(
c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.330e+06, tolerance: 9.129e+03
  model = cd_fast.enet_coordinate_descent(
c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.367e+06, toleranc

------------------ FASE 2 -------------------
Correlaciones Finales:  [0.9725819476985545, 0.9747155427618888, 0.9630098339827987, 0.9707201560468381]
Huerfanos:  (0, 176)
Longitud Grupo 0 :  161 
Longitud Grupo 1 :  195 
Longitud Grupo 2 :  204 
Longitud Grupo 3 :  159 
Grupos Definitivos:  4
R2:  0.9725819476985545
R2:  0.9747155427618888
R2:  0.9630098339827987
R2:  0.9707201560468381


c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.181e+06, tolerance: 3.009e+04
  model = cd_fast.enet_coordinate_descent(
c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.698e+06, tolerance: 3.060e+04
  model = cd_fast.enet_coordinate_descent(
c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.789e+06, toleranc

In [8]:
definitive_groups[0]

,ID_LOTE,DIAS_EN_EMERGER,DIAS_EN_EMERGER_A_FLORECER,DIAS_EN_FLORECER_A_COSECHAR,POBLACION_20DIAS_AJT,ALTURA_LOT,ContEnfQui_Emer_Flor,ContEnfQui_Flor_Cose,ContMalMec_Siem_Emer,ContMalMec_Emer_Flor,...,CAP_ENDURE_RASTA,MOTEADOS_RASTA,MOTEADOS_MAS70cm._RASTA,OBSERVA_EROSION_RASTA,OBSERVA_MOHO_RASTA,OBSERVA_RAICES_VIVAS_RASTA,OBSERVA_HOJARASCA_MO_RASTA,SUELO_NEGRO_BLANDO_RASTA,CUCHILLO_PRIMER_HTE_RASTA,CERCA_RIOS_QUEBRADAS_RASTA
1,4222,6,48,79,70000,17,0,0,0,0,...,0,0,0,0,0,1,0,0,1,0
2,4254,5,49,81,59700,18,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,3643,5,46,84,65000,9,0,0,0,0,...,0,1,0,0,0,1,0,0,1,1
4,3947,4,49,90,60000,7,1,0,0,0,...,0,0,0,0,0,1,0,0,1,1
5,2275,6,47,84,55000,8,0,0,0,0,...,1,1,0,0,0,1,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157,2488,5,45,81,72000,10,0,0,0,0,...,1,0,0,0,0,1,1,0,1,1
158,2900,4,47,80,62000,9,0,0,0,0,...,0,1,0,0,0,1,0,0,1,0
159,2693,6,53,84,60000,10,0,0,0,0,...,1,0,0,0,0,1,0,0,0,0
160,3937,5,47,90,65000,6,0,0,0,0,...,1,1,0,0,0,1,0,0,0,1


In [9]:
print("Grupos Definitivos: ", len(definitive_groups))

# Modelos Finales
list_final_models = []
for i in range(len(definitive_groups)):
    model_final, r2_final, yhat_final = LinearRegession(definitive_groups[i],0.1, 0.97).CalcularModeloLR()
    print("R2: ", r2_final)
    list_final_models.append(model_final)
    

grupos_finales = definitive_groups.copy()



Grupos Definitivos:  4
R2:  0.9725819476985545
R2:  0.9747155427618888
R2:  0.9630098339827987


c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.181e+06, tolerance: 3.009e+04
  model = cd_fast.enet_coordinate_descent(
c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.698e+06, tolerance: 3.060e+04
  model = cd_fast.enet_coordinate_descent(
c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.789e+06, toleranc

R2:  0.9707201560468381


c:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:647: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.115e+06, tolerance: 3.397e+04
  model = cd_fast.enet_coordinate_descent(


In [21]:
# Construimos datset de Trainign para etapa de clasificación
# ============================================================
final_datset_join = BuildGroupsQuality(grupos_finales)
print(final_datset_join.shape)


(719, 176)


In [7]:
dataset_ditancia = final_datset_join.drop(["RDT_AJUSTADO"], axis=1)
dataset_ditancia

,DIAS_EN_EMERGER,DIAS_EN_EMERGER_A_FLORECER,DIAS_EN_FLORECER_A_COSECHAR,POBLACION_20DIAS_AJT,ALTURA_LOT,ContEnfQui_Emer_Flor,ContEnfQui_Flor_Cose,ContMalMec_Siem_Emer,ContMalMec_Emer_Flor,ContMalMec_Flor_Cose,...,MOTEADOS_RASTA,MOTEADOS_MAS70cm._RASTA,OBSERVA_EROSION_RASTA,OBSERVA_MOHO_RASTA,OBSERVA_RAICES_VIVAS_RASTA,OBSERVA_HOJARASCA_MO_RASTA,SUELO_NEGRO_BLANDO_RASTA,CUCHILLO_PRIMER_HTE_RASTA,CERCA_RIOS_QUEBRADAS_RASTA,Grupo
0,6,48,79,70000,17,0,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0
1,5,49,81,59700,18,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
2,5,46,84,65000,9,0,0,0,0,0,...,1,0,0,0,1,0,0,1,1,0
3,4,49,90,60000,7,1,0,0,0,0,...,0,0,0,0,1,0,0,1,1,0
4,6,47,84,55000,8,0,0,0,0,0,...,1,0,0,0,1,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
714,4,47,85,55000,11,0,0,0,0,0,...,1,0,0,0,1,0,0,1,0,3
715,5,51,83,59800,16,0,0,0,0,0,...,0,0,0,0,1,1,0,1,0,3
716,4,47,87,55000,4,0,0,0,0,0,...,1,0,0,0,1,0,0,1,0,3
717,6,49,84,65750,17,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,3


In [8]:
# Agrgamos los limites para las predicciones reales
df_limite = final_datset_join.copy()
df_limite["RDT_mas15"] = df_limite.RDT_AJUSTADO*0.15 + df_limite.RDT_AJUSTADO
df_limite["RDT_menos15"] = df_limite.RDT_AJUSTADO - df_limite.RDT_AJUSTADO*0.15 

In [10]:
limites_15 = df_limite[["RDT_AJUSTADO","RDT_mas15","RDT_menos15"]]
limites_15

,RDT_AJUSTADO,RDT_mas15,RDT_menos15
0,7303.72,8399.2780,6208.1620
1,6169.19,7094.5685,5243.8115
2,4767.44,5482.5560,4052.3240
3,6674.42,7675.5830,5673.2570
4,2383.72,2741.2780,2026.1620
...,...,...,...
714,5053.49,5811.5135,4295.4665
715,6105.33,7021.1295,5189.5305
716,1158.14,1331.8610,984.4190
717,6310.47,7257.0405,5363.8995


In [39]:
df_norm = NormalizeViewMinable(dataset_ditancia.values[:,:-1],valMin, dataRange)
print(df_norm.shape)


(719, 174)


In [49]:
# df: Datasaset distancia que incluye los atributos con los grupos
def qualityFunction(df_norm, wi,df, lista_modelos, limites_15):
    #print("longitud df_norm: ",len(df_norm))
    #print("Longitud wi: ", len(wi))
    y_predicho = []
    contador = 0
    for i in range(len(df_norm)):
        vrf = df_norm[i] 
        minDep = sys.float_info.max
        posMinDep = 0
        for j in range(len(df_norm)):
            if i != j:
                ri = wi* np.power((df_norm[j] - vrf), 2)
                dE = np.sum(ri)
                if dE < minDep:
                    posMinDep=j
                    minDep = dE
        #print(posMinDep)
        modelo = df.values[posMinDep][-1]
        #print("Modelo: ", int(modelo))
        # caculamos la predicción de c/d registro con el modelo seleccionado
        y_pred = lista_modelos[int(modelo)].predict(df.values[:,:-1][i].reshape(1,-1))[0]
        #y_predicho.append(y_pred)
        
        
        if (y_pred >= limites_15.values[i][2]) & (y_pred <= limites_15.values[i][1]):
            contador = contador + 1
            #print("Entra")
        
    qs = contador/len(df_norm)
    #print("La Calidad es:  ", qs)
    return qs




def ImprovisationGBHS(PAR, hmn, nc,dataset_normalizado , lmp, HMRC,df,lista_modelos, limites_15):

    df_norm = dataset_normalizado

    # Se genera la memoria Armonica - diferentes tamaños
    np.random.seed(123)
    MA = GenerateArmonyMemory(df_norm,hmn,dataset_ditancia,lista_modelos, limites_15)
    P=len(MA[0]) 
    
    
    curvaFitnes = []
    vectorIteration= []
    for i in range (lmp):
        pesosAleatorios = np.random.rand(P-1)
        for j in range(P-1):
            Aleatorio1 = random.random() 
            if (Aleatorio1 < HMRC):
                pma = random.randint(0, hmn-1)
                pesosAleatorios[j]= MA[pma][j]

                Aleatorio2 = random.random()
                if Aleatorio2 < PAR:
                    pesosAleatorios[j] = MA[0][j]
            
            else:
                Aleatorio3 = random.random()
                if Aleatorio3 < nc/P:
                    Aleatorio4 = 0
                else:
                    Aleatorio4 = Aleatorio3/(P-nc)
                
                pesosAleatorios[j] = Aleatorio4
        
        
        print("Suma pesos aleatorios: ", pesosAleatorios.sum())
        # Normalización de los pesos
        sum = pesosAleatorios.sum()
        WN = pesosAleatorios / sum

        fitnes = qualityFunction(df_norm, WN,df,lista_modelos, limites_15)
        print("Funcion de calidad: ", fitnes)



        # Remplazo
        if MA[hmn -1][P-1] < fitnes:
            new_register = np.append(WN, fitnes)
            MA[hmn-1] = new_register
            #print("------------------------------------")
            MA.sort(key=lambda x: x[-1], reverse=True)
        

        
        curvaFitnes.append(MA[0][P-1])
        vectorIteration.append(MA[0])
        

    
    
    print("Valor de la curva en la ultima poisción: ",curvaFitnes[-1])
    '''
    dicc = {"vector":vectorIteration,
                "Fitnes": curvaFitnes}
    

    df_new = pd.DataFrame(data=dicc)
    df_new.to_csv(f"ResultadosImprovisacion/GBHS.csv")
    dicc = dict()
    '''
    return [curvaFitnes[-1], vectorIteration[-1]]

In [41]:
#
#  Prueba del vector de pesos y la funcion de calidad

wi = GenerateWeightVector(174)
Qs = qualityFunction(df_norm, wi,dataset_ditancia, list_final_models,  limites_15)

print("Calidad: ", Qs)

Calidad:  0.545201668984701


In [42]:
MA = GenerateArmonyMemory(df_norm,10,dataset_ditancia,list_final_models, limites_15)

In [43]:
MA

[array([6.25152975e-03, 9.03109658e-03, 8.69977562e-03, 3.01461489e-03,
        7.22760525e-03, 9.58458244e-03, 6.68598503e-03, 2.80924476e-03,
        6.77357824e-03, 8.85040837e-03, 2.50309501e-03, 1.66585732e-03,
        3.37102042e-03, 5.08146831e-03, 4.82830973e-03, 1.65344518e-03,
        9.82226822e-03, 5.70766342e-03, 7.41529352e-03, 6.30299795e-04,
        6.03040036e-03, 7.71888377e-03, 1.54174240e-03, 1.02828798e-02,
        9.26099125e-03, 4.33816970e-03, 7.46546671e-04, 5.15816844e-03,
        9.87684123e-03, 9.19517283e-03, 8.54181760e-03, 6.93721648e-03,
        8.45761375e-03, 9.77097382e-03, 6.04600339e-03, 2.75945305e-03,
        8.46540433e-03, 2.74659651e-03, 7.29573078e-04, 3.82005103e-03,
        9.98998196e-03, 1.53811750e-03, 3.10004217e-03, 3.43282893e-03,
        3.28277586e-03, 2.59018791e-03, 4.09691827e-03, 1.00612259e-02,
        8.94336873e-03, 2.25175237e-04, 1.02003780e-02, 4.29018682e-03,
        3.55409627e-03, 3.76960821e-03, 6.86216052e-03, 9.342165

In [50]:
metric, vector_pesos =  ImprovisationGBHS(0.4, 20, 50, df_norm, 100, 0.85, dataset_ditancia,list_final_models, limites_15)

Suma pesos aleatorios:  0.904044258148516
Funcion de calidad:  0.5855354659248957
Suma pesos aleatorios:  1.0069929178045625
Funcion de calidad:  0.5952712100139083
Suma pesos aleatorios:  0.9524564811326522
Funcion de calidad:  0.5591098748261474
Suma pesos aleatorios:  0.936666913684646
Funcion de calidad:  0.5869262865090403
Suma pesos aleatorios:  0.8282107549313582
Funcion de calidad:  0.5841446453407511
Suma pesos aleatorios:  0.8556980772588425
Funcion de calidad:  0.5855354659248957
Suma pesos aleatorios:  0.925318955210275
Funcion de calidad:  0.5938803894297635
Suma pesos aleatorios:  0.9954947617613146
Funcion de calidad:  0.5841446453407511
Suma pesos aleatorios:  0.9487713928978982
Funcion de calidad:  0.5757997218358831
Suma pesos aleatorios:  0.947308894506762
Funcion de calidad:  0.5674547983310153
Suma pesos aleatorios:  0.9450604286555273
Funcion de calidad:  0.5813630041724618
Suma pesos aleatorios:  0.9714368441710792
Funcion de calidad:  0.56884561891516
Suma pesos

: 

In [19]:
sum(vector_pesos)

1.625869262865091